## Classes, Objects, self, `__init__`, and Variables

## why do we even need `object`??? 

~~ object  in an instace of a class....~~

Le't forget `car`, `dog`,  `student` etc.

suppose we're working with a machine learning model. 

In [ ]:
from sklearn.ensemble import RandomForestClassifier 

model = RandomForestClassifier(
    n_estimator = 300, 
    max_depth = 31, 
    random_state = 42
)
model.fit(X_train, y_train)
prediction = model.predict(X_test)

here, what is `model`. It's not just a variable, 
It's an object which represent the state + behavior.

* Model remeber the `configuration`, `behavior`
```
RandomForest model
│
├── Configuration
│   ├── n_estimators = 300
│   ├── max_depth = 12
│   └── random_state = 42
│
├── Learned state
│   ├── estimators_
│   ├── feature_importances_
│   └── ...
│
└── Behavior
    ├── fit()
    ├── predict()
    ├── score()
    └── ...
```

object is an instace of a class.. 

better to say... 

An `object` packages the `state` + `operations` that works on that state. 

OOP give us a natural unit. 
```
             MODEL OBJECT
        ┌────────────────────┐
        │                    │
        │   STATE            │
        │   parameters       │
        │   learned data     │
        │                    │
        │   BEHAVIOR         │
        │   fit()            │
        │   predict()        │
        │   score()          │
        │                    │
        └────────────────────┘
```

## What is a class then ?

- A reusable template, which groups data(`variables`) and behaviors (`functions`) into a single logical package. 

- Class is defination of behavior and structure. 


## Object have their own states:
```
model_a = RandomForestClassifier(n_estimators=100)

model_b = RandomForestClassifier(n_estimators=500)
```
both are coming from same class, but have different states .

```
model_a.fit(X1, y1)
model_b.fit(X2, y2)
```
They also have different learned states. 

## `Self`

In [ ]:
class SomeModel:

    def predict(self, X):
        ...

model.predict(X_test)


## This should automatically be understood as 
SomeModel.predict(model, X_test)
# when we've other model2 object.

SomeModel.predict(model_2, X_test)


Why does `self` exist??
- Because the method (`predict`) need to know which `model's state` it is operating on. 

In [10]:
## example 
class MeanPredictor:
    def fit(self, y):
        self.mean_ = sum(y) / len(y)
        return self
    def predict(self, X):
        return [self.mean_] * len(X)
    

In [ ]:
model = MeanPredictor()
model.fit([20,30,40])
model.predict([1])

## Here predict is accessing self.mean_ = 30

[30.0]

In [15]:
model2 = MeanPredictor()
model2.fit([100, 200, 300])
model2.predict([600])

# Here, we're talking about self.mean_ = 200, basically context is chaned to model2 now. 

[200.0]

In [ ]:
model.fit([20,30,40])
# this is equivalent to below (for understanding purpose)
## basically self is the `model` itself.
MeanPredictor.fit(model, [20, 30, 40])


* Basically python knows which model/model2 to fit/predict through `self` keyword. 
- Infact `self` is not keyword, just a convention followed, you can write `this` or anything. 

## What is `__init__()` then.

- Now that we've an object, we want it to remember its initial states. 


In [ ]:
# so while writing
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12
)
# the newly created object will remember 
n_estimators = 300
max_depth = 12

** this is why `__init__()` is commonly used for. 
* `Initiliziing the states`

In [ ]:
class Model:
    def __init__(self, learning_rate, epochs):
        self.learning_rate = learning_rate
        self.epochs = epochs

model = Model(0.01, 100)

* Thus the Model object now becomes:- 
```
model
│
├── learning_rate → 0.01
├── epochs → 100
│
├── fit()
└── predict()
```

### why does  `__init__()` has `self`??
- so writing `model = Model(0.01)` is equivalent to understanding 
`Model.__init__(model, 0.01)`, 
- So, everything will be behaved as per the states of `model`. 
- That's what `self` is referring to `model` here.

## Instance Variables v/s Class Variables. 

In [ ]:
class Experiment:

    framework = "Python"

    def __init__(self, name):
        self.name = name

In [ ]:
xyz = Experiment('abc')
pqr  = Experiment('random')

* Framework is `class variable`, name is `instance variable` referring only to that instance of any class object. 
* 'abc` and 'random' are instance variables limited only for the scope of their respective object. 


### Why use class variables?
- when the value is shared by all the instances. 
- eg - `timeout = 20`

To remember :- 

`Class` - Shared defination + shared behavior   
`Object` - 1 particular instance + its own state  
`self` - Which particular instance we're talking about.  

In [16]:
class Model:

    framework = "sklearn"

    def __init__(self, name):
        self.name = name


m = Model("Fraud Detector")

print(m.__dict__)
print(Model.__dict__.keys())

{'name': 'Fraud Detector'}
dict_keys(['__module__', 'framework', '__init__', '__dict__', '__weakref__', '__doc__'])


* we can see that the `name` belong to the object instance, but the `framework` is defined on the class. 

In [18]:
class Config:
    timeout = 40

a = Config()
b = Config()
a.timeout = 99
print(a.timeout)
print(b.timeout)

print(Config.timeout)

99
40
40


* this is because, `attribute lookup` is first checked up in the instance .
* LEGB, local, Enclosing, Global, concept.. 

# Interview Focused qns:- 

Explain `self`
- it refers to the current object...
- better explain with example. 
- `model.predict(x)`, here, python passes model as the first argument to the method. allowing the method(predict), to access that instance's state through `self`. 

### Mental model
```
             CLASS
              │
      ┌───────┴────────┐
      │                │
   methods         class attrs
      │                │
      └───────┬────────┘
              │
       ┌──────┴──────┐
       ▼             ▼
    object A      object B
       │             │
   own state      own state
       │             │
      self          self
```

# Instance, Class & Static methods + @Property

This section will help us understand, `who should this method belong to`. 
 One object, whole class or neither?

- the main difference b/w them will be  `what data they access to and how are they called.`

## 1 Instance Method. 

* work on this object. 

- are bound to a specific object instance and can modify its unique state only 

```
1. model_a.predict(X)
2. model_b.predict(X)
```
* Here, same method `predict` is being used, but each call operate on a different model.

* We're telling `predict` to work on the respective object model. 


* _ So use `instance method ` when the operation needs a particular object's state. 

 # !
 Why do we need them, 

And when do we need them all. 

why not one only for everything..... 

## 2 Class Method
## 3 Static Method

### why we need  them all....
* Imagine we're building a Bank Account System. And we need these different method working as team to handle different responsibility. 


In [ ]:
class BankAccount:
    interest_rate  = 0.03 # shared by all accounts

    def __init__(self, owner, balance):
        self.owner = owner # unique to this account
        self.balance = balance # unique to this account

    # *---- Instance Method --- 
    # Need: Modify this person's money only. 
    def withdraw(self, amount):
        if amount <= self.balance:
            self.balance -= amount
            print(f"Withdrew ${amount}. New balance: ${self.balance}")

    

    # * ----- Class Method ---- 
    # Need: A factory to create a specific type of account with standard defaults. 
    @classmethod
    def create_child_account(cls, owner):
         # starting a child account with class blueprint (cls)
         # ! owner = owner, balance = 10
         # this `cls` is same as `self` for specific object. 
         return cls(owner, balance = 10)

    # * --- Static Method --- 
    def is_valid_currency(currency_code):
        # Purity utility check - true/False
        return currency_code in ["USD", "EUR", "GBP"]
    

* static Method - `BankAccount.is_valid_currency('USD')`. 
This checks the currency validity, even before we start. 

* Class method - `kids_acc = BankAccount.create_child_account("Leo")`
Builds a specilized kids' account

* Instance Method - `kids_acc.withdraw(6)`, spends money from that account .

__ Other examples ___

In [ ]:
class Employee:
    # Class Attribute: Base standard working hours for everyone
    standard_work_hours = 40 

    def __init__(self, name, hourly_rate):
        # Instance Attributes: Unique to each employee 
        self.name = name
        self.hourly_rate = hourly_rate

    # *--- 1. INSTANCE METHOD ---
    # Need: Accesses unique data (hourly_rate) to calculate one specific person's pay.
    def calculate_weekly_pay(self):
        return self.hourly_rate * self.standard_work_hours

    # *--- 2. CLASS METHOD ---
    # Need: A factory to build a preset object when you only have a raw text string.
    @classmethod
    def from_csv_string(cls, csv_text):
        # Parses a string like "Alice,25" into separate variables
        name, rate_str = csv_text.split(",")
        # Creates and returns the new Employee object using 'cls'
        return cls(name, float(rate_str))

    # *--- 3. STATIC METHOD ---
    # Need: A simple office rule check. It doesn't need to know who the employee is.
    # This is blind to everything the object has. ...(no object attribute information.)
    @staticmethod
    def is_work_day(day_name):
        # Pure logic: returns True for weekdays, False for weekends
        return day_name.lower() not in ["saturday", "sunday"]


### How you use them together. 

In [21]:
 # * 1. Static method. 
# No employee object exists yet. 
if Employee.is_work_day('Monday'):
    print("office open")

 # * 2. Class method. 
# To import a new hire into the system. 
# this create an object for us. 
new_hire = Employee.from_csv_string("alan, 33")


office open


In [23]:
new_hire.__dict__

{'name': 'alan', 'hourly_rate': 33.0}

* We can see thta this `new hire` has got the attributes of the Employee Class. 
* We created a new object, and `cls` tagged it to the the class attributes. 

* Now that the object is created, we can use the instance method, find his pay.

In [24]:
alan_pay = new_hire.calculate_weekly_pay()
print(f"{new_hire.name} earned ${alan_pay}") 

alan earned $1320.0


## Why one method cannot do it all by itself?? 

* `is_work_day`: can't be an instance method, because HR just need to check if the day is workday, `before any employees are loaded` into the system. 

* `from_csv_string` can't be a normal instance method, because you can't call an instance method on object which `hasn't been created yet`. 

* `calculate_weekly_pay` can't be static method, becoz a static method is `blind to alan's hourly rate`. 

## Qns:

1. Static method is faster. ?
- Not major performance advantage, rather use it when the operation logically belongs to the class, but doesn't need instance or class state. 


## @Property

* this decorator turns a `method` into a `read only variable (attribute)`

* Allows to call method without `()`

In [ ]:
## Let's understand this through example. 

class Thermometer:
    def __init__(self, celcius_value):
        self.celsius = celcius_value

    @property
    def fahrenheit(self):
        print('calculating live..')
        return (self.celsius * 9/ 5) + 32
    


In [30]:
t = Thermometer(25)
print(t.fahrenheit)

calculating live..
77.0


* No parenthesis `()` at the end of fahrenheit.
* without `@property` we would have to write `t.fahrenheit()`

In [39]:
# If you change the celsius, the property updates automatically
t.celsius = 37
print(t.fahrenheit)  


calculating live..
98.6


* By default, `@property` is `read-only`. doing `t.farenheit = 200`, will throw error. 

* If you want to allow the changes,  but need to validate them first, you can pair it with  `.setter`.

In [41]:
class SmartWallet:
    def __init__(self, balance):
        self._balance = balance

    @property
    def balance(self):
        return self._balance

    # ! Setter for the check. 
    @balance.setter
    def balance(self, new_amount):
        if new_amount < 0:
            print("Error: balance can't be negative")
        else:
            self._balance = new_amount

In [44]:
wallet = SmartWallet(49)
print(wallet.balance)

49


In [45]:
wallet.balance = 100
print(wallet.balance)

100


In [ ]:
wallet.balance = -20


Error: balance can't be negative


* Setter allows us to check any invalid modifications. 

* Other examples of  `property usage`. 

 `df.shape`

In [ ]:
@classmethod
def ...

@staticmethod
def ...

@property
def ...

## These decorator modify how the function behave when accessed. 

Questions:- 
1. Can a static method access `self`?
- Not automatically, if we need a   `self` then it isn't really a `static method` right...

## Putting everything together

In [48]:
class Experiment:

    framework = "sklearn"

    def __init__(self, name, learning_rate):
        self.name = name
        self.learning_rate = learning_rate

    @classmethod
    def from_config(cls, config):
        return cls(
            config["name"],
            config["learning_rate"]
        )

    @staticmethod
    def validate_learning_rate(value):
        return value > 0

    @property
    def description(self):
        return (
            f"{self.name} "
            f"(lr={self.learning_rate})"
        )

    def train(self, X, y):
        print(
            f"Training {self.name}"
        )

In [ ]:
# # ! Instance method, accessing the `self`
experiment = Experiment()
experiment.train(X, y)
# or understand this way Experiment.train(self = experiment,X, y)

# # ! Class method. 
Experiment.from_config(config)

# # ! this needs neither, so static method, 
Experiment.validate_learning_rate(0.01)

# # ! Computed  attribute. :so  property. 
experiment.description

# Dunder Method. or `magic methods`

* Dunder means `double score`. `__...__`

* We never call dunder method directly, instead we use actions like `+`, `print()`, `len()`. and python secretly translate them into dunder method behind the scenes. 

`print(xyz)` -> `obj.__str__()`

`len(obj)` -> `obj.__len__()`

`a + b` --> `a.__add__(b)`

In [58]:
## real world examples. 

class ShoppingCart:
    def __init__(self):

        self.items = []

    def __str__(self):
        return f"Cart has: {', '.join(self.items)}"

    def __len__(self):
        return len(self.items)

    def __add__(self, new_item):
        self.items.append(new_item)
        return self


In [61]:
cart = ShoppingCart()
#! + trigger __add__ secretly. 
cart = cart + "Apple"
cart = cart + "Milk"

#! len trigger __len__
print(len(cart))

#! print trigger __str__
print(cart)

2
Cart has: Apple, Milk


* Some common Dunders are. 
- `__init__`,  
 `__str__`,   
 `__len__`,  
  `__eq__`.

### `__str__`

In [62]:
class Experiment:

    def __init__(self, name, score):
        self.name = name
        self.score = score

In [63]:
experiment = Experiment("Fraud Detection", 0.94)

print(experiment)

* This looks ugly right... don't know what is happening.. 

* Same can be written the following way. 

In [64]:
class Experiment:

    def __init__(self, name, score):
        self.name = name
        self.score = score

    def __str__(self):
        return f"{self.name}: {self.score:.2%}"

In [ ]:
experiment = Experiment("Fraud Detection", 0.94)
print(experiment)

Fraud Detection: 94.00%


* So when we do `print`, `__str__` is triggered, and now we modified it as per our readability. 

### `__repr__`

Now imagine you're debugging a production ML pipeline.
Then we would want something useful for the developer.. 

that's where `__repr__` comes in. 

In [82]:
class Experiment:

    def __init__(self, name, score):
        self.name = name
        self.score = score
    def __repr__(self):
        return (
            f"Experiment("
            f"name={self.name!r}, "
            f"score={self.score!r})"
        )

In [85]:
experiment = Experiment("Fraud Detection", 0.94)
experiment


Experiment(name='Fraud Detection', score=0.94)

## diff b/w `__str__` and `__repr__`

* main diff is `__str__` is  `human friendly` 
* `__repr__` is `debugger friendly`. 

* In an interactive enviroment, it generally use `__repr__`. 

* If you don't define `__str__`, then python fallback to `__repr__`  for string represntation i.e., print statement. 


In [79]:
class Experiment:

    def __init__(self, name, score):
        self.name = name
        self.score = score
    def __str__(self):
        return f"Human friendly __str__ for print: {self.name}"
    
    def __repr__(self):
        return (
            f"Experiment("
            f"name={self.name!r}, "
            f"score={self.score!r})"
        )

In [80]:
experiment = Experiment("Fraud Detection", 0.94)
print(experiment)
experiment

Human friendly __str__ for print: Fraud Detection


Experiment(name='Fraud Detection', score=0.94)

`[]` operator -> `__getitem__` 

Eg. `list[9]`

* For this python invokes `__getitem__(9)`

In [ ]:
df['age']
df.iloc[2]
tensor[0]

## All of these ultimately transition to 
object.__getitem__(key)



### `__iter__()`

 Similarly `__iter__()`, to go to next item while using `list`, `string`, `dictionary`. 

 * think of `__iter__()` as a `button` that `starts a conveyer belt`

In [111]:
# Let's try to implement __iter__
class Team:
    def __init__(self, team_name, player_list):
        self.team_name = team_name
        self.player_list = player_list 


my_team = Team("Lakers", ["LeBron", "AD", "Austin", "DLo"])
for player in my_team:
    print(player)

TypeError: 'Team' object is not iterable

* `TypeError: 'Team' object is not iterable`
- Getting this bcoz, object type `class` is not iterable, generally `list`, `dict` are iterables. 

* to fix this, we must implement the `__iter__()` magic method. 

In [114]:
# Let's try to implement __iter__
class Team:
    def __init__(self, team_name, player_list):
        self.team_name = team_name
        self.player_list = player_list 

    def __iter__(self):
        for player in self.player_list:
            yield player

my_team = Team("Lakers", ["LeBron", "AD", "Austin", "DLo"])
for player in my_team:
    print(player)

LeBron
AD
Austin
DLo


* what is `yield` doing here, 
* while `return` completely stops the function and kills it, 
* `yield` passes a value out of the loop, `pauses the function`, and freezes its memory right there. 
* when the function wakes up again where it left off. 

### Why Use yield Instead of return?
1. `Memory saver`: Return loads all the data into the `RAM` at the same time. 
- `Yield` keeps `1 row at a time` in memory. 
2. Infinte Streams: - We can use it to create `infinite loop` , like clock ticking or a `live sensor data stream` without crashing your computer memory. 

## Why Dunder methods... 

In [91]:
# Consider this example. 
class Book:
    def __init__(self, title, total_pages):
        self.title = title
        self.total_pages = total_pages


story = Book("sound of mind", 311)


In [92]:
print(len(story))

TypeError: object of type 'Book' has no len()

`TypeError: object of type 'Book' has no len()`

* Suppose by writing `len(story)` we meant to get the pages count.
* This could have been prevented, if we had modified the `__len__()` to return the len of pages..

In [ ]:
class Book:
    def __init__(self, title, total_pages):
        self.title = title
        self.total_pages = total_pages

    # ! Le't modify the len to return the total page count
    def __len__(self):
        return self.total_pages


story = Book("sound of mind", 311)

print(f"Length of the class is: {len(story)}")

Length of the class is: 311


### What will happen if you don't write a dunder method,

* To make your custom objects behave exactly like python built-in-types. (string, numbers, list.)

* A unified desing like `len()` to get the size of a string, list or dictionary .
* Otherwise, one developer will write, `.get_pages()`, '.page_count()`, and third writes '.sizes()`, to get the len of the an object. 

### Can You Abuse This? (Should you modify it to do something weird?)

* Never modify a dunder method to do totally unrelated or weird things, 
* This is `bad programming`... `🥲`

## Interview Traps

1. `__str__` and `__repr__` are identical. 
- NO, `__str__` is human friendly, 
- `__repr__` is developer/debugging representation

2. `__init__` is a dunder method, so it creates the object. 
- No, `__new__` creates object, `__init__` initialize the attributes. 

#### what is `Operator Overloading` in python?
- Adding `extra responsibility` or overloading existing `symbols` 
- Eg, `+` is overloaded to handle `4+3`, ` str1 + str2`, `list1 + list2`. 


* If we compare `class A == class B`, then it will be true only if the data in them are exactly same. 

In [ ]:
class A: 
    def __init__(self):
        pass

a = A()
b = a
print(a == b)

print(id(a) == id(b))

True
True


* so here, it is checking if they sit in the same memory address. 
* we can overload this `==` equality operator to return True only if certain attribute inside them are equal. 

In [110]:
class A: 
    def __init__(self, name, age, income):
        self.name = name
        self.age = age
        self.income = income

    # ! Let's overload the equality operator to return equal income irrespective of their ages
    def __eq__(self, other):
        return self.income == other.income
        


a = A("Ram", 30, 3000)
b = A("Naresh", 50, 3000)
print(f"Equal objects: {a == b}")

print(f"Memory is same ? {id(a) == id(b)}")

Equal objects: True
Memory is same ? False


## Summarizing Dunder method:- 
`Python operation` → `protocol` → `dunder method `→ `object-specific behavior.`

More simply, 

`The Action` →`The Rulebook` → `The secret Backdoor` → `The custom Result`

# Equality and Hashing & `__hash__`
```
 Equality & Hashing
   ├── hash()
   ├── Hashable vs Unhashable
   ├── dict / set connection
   ├── __eq__ / __hash__
   └── Hash contract
   ```

Think of `hash` like `human fingerprint`.

* `Hashing` means taking any piece of data `(string, number or file)` and converting it into a `fixed-size number.`


We have memorized `dictionary keys` are hashable... 

### why does a dictionary need hashing..?


In [122]:
user_scores = {
    "Raushan": 95,
    "Amit": 87,
    "John": 91
}
# when we do this, python needs to find the value quicky, 
# ! One by one Scanning will take longer time right.. .
user_scores['John']

91

In [125]:
print(hash('John'))
# This hash value help python to locate the key efficiently

-7922365725024490070


### what is hashable?
- Object is hashable, if python can obtain a hash value from it, which remain quite stable during its lifetime and suppport equality semantics... for use in hash tables. 

In [4]:
class Klass:
    def __init__(self):
        pass
kls = xyz()

print(hash(kls))

print(id(kls))

157444235037
2519107760592


* So here, `id(kls)` is the physical location where the object live. 
* `hash(kls)` is like the `room number` or `locker combination` assigned to that object.  

### Why is list `unhashable`?
* Obvious reason, since lists are mutable so changing the entries in it, would result in mismatch of the hashvalue 

In [ ]:
x = [32,3,44]

dictt = {
    x: "list x"

}
x.append(999)


* For example, If this was allowed, then hash_key of `x` is initally `1000` and on appending another value it changed to `1001`, and thus, the value in bucket `1000` is permanently lost, as the new locker number has changed. 

* so the list is physically `trapped at 1000`. 

* since the key has changed during the time. 
* so we keep `immutable` objects as hash value.

##### So the common immutable objects `int, float, str, tuple, frozenset` can be hashable. 

In [14]:
a, b = 3, 3
# ! iF 
print(a == b)
# ! THEN.. 
print(hash(a) == hash(b))

True
True


* But reverse is not guaranteed 
* this is called `hash collision`. 

In [15]:
a = 1
b = True
print(hash(a), hash(b))

1 1


* Same hash value.. 
this is addreseed in python using strategy called `open addressing` (`linear probing`)
* loook for the very `next empty slot`


Hashing matters in production Data Science code. 
- caching
- memoizatoin
- 

# Inheritance, `super()` & MRO

So we often remember `inheritance` as   
`class Child(Parent):` and `super().__init__()`.

### Why do use `inheritance`?

In [16]:
# Consider this example. 
class Model:
    def train(self):
        print("traininigg....")
    def predict(self, X):
        print("Making prediction.....")

* Now we build some `machine learning models` say `Logistic Regression`, `RandomForest` or `NeuralNetwork`. 
* Then there we'll require all these functionality of `training` and `predicting` right... 
* so why `rewrite them into those models again...?? 🙂🙂

In [ ]:
class LogisticRegression:
    # ! Redundant
    def train(self):
        pass

    def predict(self, X):
        pass

    def find_coefficients(...):
        pass

In [17]:
# so here we're writing the functionality of training and predicting again.... 

In [ ]:
## Instead we'll rewrite this as:- 
class Model:
    def train(self):
        pass
    def predict(self, X):
        pass

# ! Inheriting the functionality of Model class. 
class NeuralNetwork(Model):
    pass

model = NeuralNetwork()
model.train()
model.predict(X)



### How does this work::?
* Is it plain copying... ?


In [ ]:
class NeuralNetwork(Model):

    def predict(self):
        print("Neural network prediction")

model = NeuralNetwork()
model.predict()

* So here, the model tries to find `predict` which is not present in the child class. 
* so it automatically looks for the inherited/`parent` class. 

The child simply inherits access to use it.



#### Overriding a method. 

In [ ]:
class Model:
    def predict(self):
        print("Generic Prediction")

class NeuralNetwork(Model):
    def predict(self):
        print("Neural Network Prediction... ")
        
model = NeuralNetwork()       
model.predict()

* so here, the model uses its' own class prediction. 
* Because it find the `predict` in the main class itself, so `don't look in its parent anymore.`

! this is `method overriding`, 
* the child has overriden the functionality of the parent's implementation. 

# Polymorphism

* Many + behavior.

Examples of polymorphism:- 
1. Model training and Prediction... 
- Every single machine learning models in python.. use the same two methods.. `.fit()`, and `.predict()`. 

In [ ]:
models = [
    LogisticRegression(),
    RandomForest(),
    NeuralNetwork()
]

for model in models:
    model.predict()

The name `.predict() `is polymorphed because it takes on "many forms" across your code.

* so here, each of the models have the functionality of `predict` implemented differently to predict the similar output. 
* so the` interface is same`, but `implementation different. `


Otherwise, we would have to write the following.. 

In [ ]:
if isinstance(model, LogisticRegression):

    def predict_logisticregreession():
        
elif isinstance(model, RandomForest):

    def predict_randomforest():


### `__init__`

In [3]:
class Model:

    def __init__(self, name):
        self.name = name

class NeuralNetwork(Model):
    pass

network = NeuralNetwork("ResNet")

In [4]:
network.name

'ResNet'

* so here,the child doesnot have the name, so it inherit the name from the `base` model. 

In [9]:
class Model:

    def __init__(self, name):
        self.name = name

class NeuralNetwork(Model):

    def __init__(self, name, layers):
        # ! Name is not defined.. 
        self.layers = layers

network = NeuralNetwork("ResNet", 50)

network.layers

50

In [10]:
network.name

AttributeError: 'NeuralNetwork' object has no attribute 'name'

`AttributeError: 'NeuralNetwork' object has no attribute 'name'`

* so here, we're defining the child `__init__`, which replaced the inherited inialized behavior. 

* since the child already have the `__init__`, so parent's `__init__` isn't automatically called. 

## `super()`